# Step 1: Get Features From Multiple Datasets
- Using [pybiber](https://pypi.org/project/pybiber/)

In [1]:
import os
import random
import torch
import gc
import pybiber as pb
import polars as pl
import pandas as pd
import numpy as np
from transformers import pipeline
from scipy.stats import zscore, pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, matthews_corrcoef

# HuggingFace Login
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)
# Set up transformers logging (set it to minimal logging).
from transformers import logging
logging.set_verbosity_error()

In [2]:
DEVICE = 0 if torch.cuda.is_available() else -1
FILE_PATH = 'getText/datasetsPrep'
OUTPUT_DIR = 'biberOutputs'
# 100 samples (total) chosen for comparability purposes with "Measuring the Measuring Tool" by Kour et al. (https://doi.org/10.18653/v1/2022.gem-1.35). We will do 10 x 100. This also allows for the test and validation sets to be adequately compared.
SAMPLE_SIZE = 10
random_change = 0
BATCH_SIZE = 4

In [3]:
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-deberta-v3-small", # low capacity
    "typeform/distilbert-base-uncased-mnli", # medium capacity
    "valhalla/distilbart-mnli-12-3", # higher capacity
]

# Exact mapping taken from https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html which has extracted the same from Biber and Conrad's Variation in English (https://doi.org/10.4324/9781315840888)
BIBER_LABEL_MAP = {
    "factor_1": {
        "informational, dense, precise": -1,
        "involved, interactive, affective": 1
    },
    "factor_2": {
        "non-narrative, expository, informational": -1,
        "narrative, event-focused, storytelling": 1
    },
    "factor_3": {
        "situation-dependent, context-bound, implicit": -1,
        "explicit, context-independent, elaborated": 1
    },
    "factor_4": {
        "non-persuasive, non-argumentative, neutral": -1,
        "persuasive, argumentative, modalized": 1
    },
    "factor_5": {
        "non-abstract, concrete, human-centered": -1,
        "abstract, impersonal, technical": 1
    },
    "factor_6": {
        "compressed, dense, clause-poor": -1,
        "elaborated, expanded, clause-rich": 1
    }
}

# Flatten Biber label map (faster inference).
all_labels = []
label_to_factor = {}

for factor, description in BIBER_LABEL_MAP.items():
    for label in description.keys():
        all_labels.append(label)
        label_to_factor[label] = factor

# Using different prompt templates increases robustness.
TEMPLATES = ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."]

In [4]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [5]:
# Load all data.
list_of_dfs = []
for folder in os.listdir(f'./{FILE_PATH}'):
    if os.path.isdir(f'./{FILE_PATH}/{folder}'):
        for file in os.listdir(f'./{FILE_PATH}/{folder}'):
            if file.endswith(".csv") and 'train' in file:
                temp_file_path = f'./{FILE_PATH}/{folder}/{file}'
                temp_tag = file.replace('_train.csv', '')
                temp_df = pl.read_csv(temp_file_path)
                temp_df = (
                    temp_df
                    .with_row_index("index_num") # , offset=1) if you want to start index from 1
                    .with_columns(
                        (pl.lit(temp_tag) + "_" + pl.col("index_num").cast(pl.Utf8)).alias("doc_id")
                    )).select(['text', 'doc_id'])
                list_of_dfs.append(temp_df)

combined = pl.concat(list_of_dfs, how="vertical")
assert combined.select(pl.col("doc_id").n_unique()).item() == combined.height, "There should be no duplicates in the dataset."

In [6]:
# Remove invalid data.
combined = combined.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(combined.group_by("tag").len())
print(f"Total Texts Before Empty String Removal: {len(combined)}")

combined = combined.with_columns(pl.col("text").str.strip_chars().alias("text")).filter(pl.col("text").is_not_null() & (pl.col("text") != ""))

combined = combined.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
print(combined.group_by("tag").len())
print(f"Total Texts After Empty String Removal: {len(combined)}")

combined = combined.with_columns(
    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
)
combined = combined.to_pandas()

shape: (10, 2)
┌────────────────────────────────┬────────┐
│ tag                            ┆ len    │
│ ---                            ┆ ---    │
│ str                            ┆ u32    │
╞════════════════════════════════╪════════╡
│ clinicalDialogueSummarizations ┆ 2521   │
│ yahoo                          ┆ 61152  │
│ clinc150                       ┆ 16589  │
│ simSUM                         ┆ 7000   │
│ dementiaAudio                  ┆ 383    │
│ medicalAbstracts               ┆ 10106  │
│ syntheticCareHomeNurseNotes    ┆ 4047   │
│ banking77                      ┆ 9147   │
│ atis                           ┆ 3484   │
│ huffPostNews                   ┆ 146668 │
└────────────────────────────────┴────────┘
Total Texts Before Empty String Removal: 261097
shape: (10, 2)
┌────────────────────────────────┬────────┐
│ tag                            ┆ len    │
│ ---                            ┆ ---    │
│ str                            ┆ u32    │
╞════════════════════════════════╪════════

In [7]:
# Randomly sample from dataframe.
temp_df = (combined.groupby("tag")).apply(lambda x: x.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE + random_change))
temp_df = pl.from_pandas(temp_df)

In [8]:
# Set up df for use.
df = temp_df.select(["doc_id", "text"])

# Light preprocessing to strip extra whitespace.
df = df.with_columns(
    pl.col("text")
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .str.replace_all(r"^\s*-\s*", "") # Remove dashes at the beginning of texts.
    .str.replace_all(r"^\s*\d+\.\s*", "") # Remove numbers in 1., 2., 3. format at the beginning of the text. 
)

pybiber_pipeline = pb.PybiberPipeline(model="en_core_web_sm")
features, tokens = pybiber_pipeline.run(df, return_tokens=True)
features = features.with_columns(pl.col("doc_id").str.split("_").list.get(0).alias("category"))
# Full feature list can be found here: https://browndw.github.io/pybiber/feature-categories.html
print(f" -------- Features-------- ")
print(features)

# Statistical analysis and visualization
analyzer = pb.BiberAnalyzer(features, id_column='category')

# Multi-Dimensional Analysis - see https://browndw.github.io/pybiber/biber-analyzer.html#comparison-with-bibers-original-dimensions for factor mapping
# Explanation of the factor mapping to dimensions can be found here: https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html
'''
Factor 1: Involved vs. Informational Production (negative to positive)
Factor 2: Narrative vs. Non-narrative Concerns (negative to positive)
Factor 3: Explicit vs. Situation-dependent Reference (negative to positive)
Factor 4: Overt Expression of Persuasion (negative to positive)
Factor 5: Abstract vs. Non-abstract Information (negative to positive)
Factor 6: On-line Informational Elaboration (negative to positive)
'''

analyzer.mda_biber()
print(f" -------- MDA Loadings -------- ")
print(analyzer.mda_loadings)
print(f" -------- MDA Dimension Scores -------- ")
print(analyzer.mda_dim_scores)

[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_30_that_obj', 'f_32_wh_obj', 'f_47_hedges']


 -------- Features-------- 
shape: (100, 69)
┌─────────┬─────────┬─────────┬─────────┬─────────┬───┬────────┬────────┬────────┬────────┬────────┐
│ doc_id  ┆ f_01_pa ┆ f_02_pe ┆ f_03_pr ┆ f_04_pl ┆ … ┆ f_64_p ┆ f_65_c ┆ f_66_n ┆ f_67_n ┆ catego │
│ ---     ┆ st_tens ┆ rfect_a ┆ esent_t ┆ ace_adv ┆   ┆ hrasal ┆ lausal ┆ eg_syn ┆ eg_ana ┆ ry     │
│ str     ┆ e       ┆ spect   ┆ ense    ┆ erbials ┆   ┆ _coord ┆ _coord ┆ thetic ┆ lytic  ┆ ---    │
│         ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆   ┆ inatio ┆ inatio ┆ ---    ┆ ---    ┆ str    │
│         ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆   ┆ n      ┆ n      ┆ f64    ┆ f64    ┆        │
│         ┆         ┆         ┆         ┆         ┆   ┆ ---    ┆ ---    ┆        ┆        ┆        │
│         ┆         ┆         ┆         ┆         ┆   ┆ f64    ┆ f64    ┆        ┆        ┆        │
╞═════════╪═════════╪═════════╪═════════╪═════════╪═══╪════════╪════════╪════════╪════════╪════════╡
│ atis_11 ┆ 0.0     ┆ 0.0     ┆ 47.6190 ┆ 0.0 

In [9]:
# Get Z-Scores from Biber analysis.
biber_dimensions = (analyzer.mda_dim_scores).to_pandas()
# Get factor columns.
factor_cols = [c for c in biber_dimensions.columns if c.startswith("factor")]
biber_dimensions[factor_cols] = biber_dimensions[factor_cols].apply(zscore)
for c in factor_cols:
    biber_dimensions[f"{c}_label"] = biber_dimensions[c] > 0
biber_dimensions = pl.from_pandas(biber_dimensions)
print(biber_dimensions)

shape: (100, 16)
┌─────────┬─────────┬─────────┬─────────┬─────────┬───┬────────┬────────┬────────┬────────┬────────┐
│ doc_id  ┆ doc_cat ┆ factor_ ┆ factor_ ┆ factor_ ┆ … ┆ factor ┆ factor ┆ factor ┆ factor ┆ factor │
│ ---     ┆ ---     ┆ 1       ┆ 2       ┆ 3       ┆   ┆ _3_lab ┆ _4_lab ┆ _5_lab ┆ _6_lab ┆ _7_lab │
│ str     ┆ str     ┆ ---     ┆ ---     ┆ ---     ┆   ┆ el     ┆ el     ┆ el     ┆ el     ┆ el     │
│         ┆         ┆ f64     ┆ f64     ┆ f64     ┆   ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│         ┆         ┆         ┆         ┆         ┆   ┆ bool   ┆ bool   ┆ bool   ┆ bool   ┆ bool   │
╞═════════╪═════════╪═════════╪═════════╪═════════╪═══╪════════╪════════╪════════╪════════╪════════╡
│ atis_11 ┆ atis    ┆ -0.5404 ┆ -0.4083 ┆ -0.1759 ┆ … ┆ false  ┆ false  ┆ false  ┆ false  ┆ false  │
│ 3       ┆         ┆ 25      ┆ 39      ┆ 49      ┆   ┆        ┆        ┆        ┆        ┆        │
│ atis_14 ┆ atis    ┆ -0.7556 ┆ -0.4597 ┆ -0.2406 ┆ … ┆ false  ┆ false  ┆ 

In [10]:
temp_df = temp_df.to_pandas()
texts = temp_df['text'].values.tolist()
doc_ids = temp_df['doc_id'].values.tolist()

In [11]:
assert len(texts) == len(doc_ids), "texts and doc_ids are not of the same length."

In [12]:
all_dfs = []
for model_name in ZERO_SHOT_MODELS:
    classifier = pipeline(
        "zero-shot-classification",
        model=model_name,
        device=DEVICE
    )

    # Initialize factor storage
    temp_factors_list = {
        'model_name': [model_name] * len(doc_ids),
        'doc_id': doc_ids,
        **{factor: [] for factor in BIBER_LABEL_MAP}
    }

    # Dictionary to accumulate scores per factor across templates
    factor_scores_accum = {factor: [0.0] * len(texts) for factor in BIBER_LABEL_MAP}

    # Loop over all templates
    for template in TEMPLATES:

        with torch.no_grad():
            outputs = classifier(
                texts,
                candidate_labels=all_labels,
                hypothesis_template=template,
                multi_label=True,
                batch_size = BATCH_SIZE
            )

            if isinstance(outputs, dict):
                outputs = [outputs]

            # Accumulate weighted scores per factor
            for j, output in enumerate(outputs):
                for label, score in zip(output['labels'], output['scores']):
                    f = label_to_factor[label]
                    weight = BIBER_LABEL_MAP[f][label]
                    factor_scores_accum[f][j] += weight * score

    # Average over templates
    n_templates = len(TEMPLATES)
    for factor in BIBER_LABEL_MAP:
        factor_scores_accum[factor] = [s / n_templates for s in factor_scores_accum[factor]]
        temp_factors_list[factor].extend(factor_scores_accum[factor])

    # Convert to DataFrame and append to all_dfs
    all_dfs.append(pd.DataFrame(temp_factors_list))

    # Free memory
    del classifier
    torch.cuda.empty_cache()
    gc.collect()

df = pd.concat(all_dfs)


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [13]:
dimension_labels_verbose = {
    "factor_1_label": "informational_vs_involved",
    "factor_2_label": "non-narrative_vs_narrative",
    "factor_3_label": "situation-dependent_vs_explicit",
    "factor_4_label": "non-persuasive_vs_persuasive",
    "factor_5_label": "non-abstract_vs_abstract",
    "factor_6_label": "compressed_vs_elaborated"
}

# Simple averaging will prevent over-confidence. 
mean_scores = df.drop(columns='model_name').groupby("doc_id").mean().reset_index()

# # Get factor columns.
factor_cols = [c for c in mean_scores.columns if c.startswith("factor")]
# # Get Z-Scores from zero-shot analysis.
mean_scores[factor_cols] = mean_scores[factor_cols].apply(zscore)
for c in factor_cols:
    mean_scores[f"{c}_label"] = mean_scores[c] > 0
mean_scores = pl.from_pandas(mean_scores)

results_cont = {}
for f in ["factor_1", "factor_2", "factor_3", "factor_4", "factor_5", "factor_6"]:
    pearson = pearsonr(biber_dimensions[f], mean_scores[f])[0]
    spearman = spearmanr(biber_dimensions[f], mean_scores[f])[0]
    mse = mean_squared_error(biber_dimensions[f], mean_scores[f])
    rmse = root_mean_squared_error(biber_dimensions[f], mean_scores[f])
    mae = mean_absolute_error(biber_dimensions[f], mean_scores[f])
    results_cont[f] = {"pearson": pearson, "spearman": spearman, "MSE": mse, "RMSE": rmse, "MAE": mae}
continuous_df = pd.DataFrame(results_cont).T
continuous_df = continuous_df.rename(index={
    "factor_1": "informational_vs_involved",
    "factor_2": "non-narrative_vs_narrative",
    "factor_3": "situation-dependent_vs_explicit",
    "factor_4": "non-persuasive_vs_persuasive",
    "factor_5": "non-abstract_vs_abstract",
    "factor_6": "compressed_vs_elaborated"
})
continuous_df['dimension'] = continuous_df.index
continuous_df = pl.from_pandas(continuous_df)
print(continuous_df)

results_bin = {}
for f in ["factor_1_label", "factor_2_label", "factor_3_label", "factor_4_label", "factor_5_label", "factor_6_label"]:
    acc = accuracy_score(biber_dimensions[f], mean_scores[f])
    prec = precision_score(biber_dimensions[f], mean_scores[f])
    rec = recall_score(biber_dimensions[f], mean_scores[f])
    f1 = f1_score(biber_dimensions[f], mean_scores[f])
    kappa = cohen_kappa_score(biber_dimensions[f], mean_scores[f])
    mcc = matthews_corrcoef(biber_dimensions[f], mean_scores[f])
    results_bin[f] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "kappa": kappa, "MCC": mcc}

classification_df = pd.DataFrame(results_bin).T
classification_df = classification_df.rename(index={
    "factor_1_label": "informational_vs_involved",
    "factor_2_label": "non-narrative_vs_narrative",
    "factor_3_label": "situation-dependent_vs_explicit",
    "factor_4_label": "non-persuasive_vs_persuasive",
    "factor_5_label": "non-abstract_vs_abstract",
    "factor_6_label": "compressed_vs_elaborated"
})
classification_df['dimension'] = classification_df.index
classification_df = pl.from_pandas(classification_df)
print(classification_df)

shape: (6, 6)
┌───────────┬───────────┬──────────┬──────────┬──────────┬─────────────────────────────────┐
│ pearson   ┆ spearman  ┆ MSE      ┆ RMSE     ┆ MAE      ┆ dimension                       │
│ ---       ┆ ---       ┆ ---      ┆ ---      ┆ ---      ┆ ---                             │
│ f64       ┆ f64       ┆ f64      ┆ f64      ┆ f64      ┆ str                             │
╞═══════════╪═══════════╪══════════╪══════════╪══════════╪═════════════════════════════════╡
│ 0.475758  ┆ 0.403552  ┆ 1.048485 ┆ 1.023956 ┆ 0.82496  ┆ informational_vs_involved       │
│ 0.186676  ┆ 0.118509  ┆ 1.626648 ┆ 1.275401 ┆ 0.967024 ┆ non-narrative_vs_narrative      │
│ 0.332671  ┆ 0.434183  ┆ 1.334657 ┆ 1.155274 ┆ 0.910484 ┆ situation-dependent_vs_explici… │
│ -0.008785 ┆ 0.006063  ┆ 2.017571 ┆ 1.420412 ┆ 1.054538 ┆ non-persuasive_vs_persuasive    │
│ 0.272843  ┆ 0.325768  ┆ 1.454314 ┆ 1.205949 ┆ 0.872557 ┆ non-abstract_vs_abstract        │
│ 0.000623  ┆ -0.066866 ┆ 1.998754 ┆ 1.413773 ┆ 1.072413

In [14]:
# Make directory for saving. 
output_file_path = f"{OUTPUT_DIR}/{RANDOM_STATE + random_change - 1}"
os.makedirs(f"./{output_file_path}/", exist_ok=True)

# Save texts used and their ids.
temp_df = pl.from_pandas(temp_df)
# Add category column. 
temp_df = temp_df.with_columns(
    pl.col("doc_id").str.extract(r"(^[^_]+)").alias("category")
)
temp_df.write_csv(f"./{output_file_path}/texts_and_ids.csv", float_precision=15)
temp_df.write_json(f"./{output_file_path}/texts_and_ids.json")

# Save biber results.
biber_dimensions.write_csv(f"./{output_file_path}/mda_dim_scores.csv", float_precision=15)
analyzer.mda_loadings.write_csv(f"./{output_file_path}/mda_loadings.csv", float_precision=15)
biber_dimensions.write_json(f"./{output_file_path}/mda_dim_scores.json")
analyzer.mda_loadings.write_json(f"./{output_file_path}/mda_loadings.json")

# Save zero-shot results.
df = pl.from_pandas(df)
df.write_csv(f"./{output_file_path}/all_model_zero_shot_classification.csv", float_precision=15)
df.write_json(f"./{output_file_path}/all_model_zero_shot_classification.json")

mean_scores.write_csv(f"./{output_file_path}/mean_model_zero_shot_classification.csv", float_precision=15)
mean_scores.write_json(f"./{output_file_path}/mean_model_zero_shot_classification.json")

# Save comparison results.
continuous_df.write_csv(f"./{output_file_path}/continuous_comparison_results_zero_vs_biber.csv", float_precision=15)
continuous_df.write_json(f"./{output_file_path}/continuous_comparison_results_zero_vs_biber.json")

classification_df.write_csv(f"./{output_file_path}/classification_comparison_results_zero_vs_biber.csv", float_precision=15)
classification_df.write_json(f"./{output_file_path}/classification_comparison_results_zero_vs_biber.json")
